# JetClass s25k — $\eta_0$ transfer every 5k steps

Read-only watch of the 4-GPU `capen-llama-att` grid on the way to **25k optimizer steps**.
**No training in this notebook.** Re-run the *Refresh* cell (and the plots below it) whenever a gpu-test shot finishes.

| Architecture | Color |
|---|---|
| $(L,D,h)=(3,256,8)$ | smaller model |
| $(L,D,h)=(5,512,16)$ | larger model |

Each run is `batch=512` per GPU × 4 GPUs → **global batch 2048**, so $N_\mathrm{jets} = 2048 \times$ steps (5k steps ≈ 10.2M jets; 25k ≈ 51.2M).
Validation is logged every 2500 steps in the Slurm `.out` files (`[val] ... val_roc_auc=`). Parquet is used when it appears (`*Dfull-s25k*.parquet`); mid-epoch val currently lives in the logs.

Transfer overlays use **5k / 10k / 15k / 20k / 25k**. A second figure plots every val checkpoint vs step so you can see the curve fill in between those ticks.

In [ ]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path

%matplotlib inline
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

REPO = Path("../..").resolve()
sys.path.insert(0, str(REPO / "src"))

from cpen.utils.transfer_plots import JETCLASS_OUT_ROOT, savefig, setup_mpl_style

setup_mpl_style()

FIG_DIR = REPO / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT = JETCLASS_OUT_ROOT
SLURM_LOG_ROOT = Path("/scratch/gpfs/BHANIN/jdezoort/cpen_runs/jetclass_runs")
LOG_GLOBS = ("jc_att_s25k_*.out", "jc_att_L5_s25k_*.out")
PARQUET_GLOB = "capen-llama-att_*Dfull-s25k*.parquet"
RUN_DIR_GLOB = "capen-llama-att_*Dfull-s25k-seed0"

MAX_STEPS = 25_000
STEP_GRID = (5_000, 10_000, 15_000, 20_000, 25_000)
GLOBAL_BATCH = 512 * 4  # per-GPU batch × n_gpus
TASK_ETA = {0: 0.05, 1: 0.25, 2: 0.5, 3: 1.0}
LOG_ARCH = {
    "jc_att_s25k": (3, 256, 8),
    "jc_att_L5_s25k": (5, 512, 16),
}

print("OUT_ROOT =", OUT_ROOT)
print("SLURM    =", SLURM_LOG_ROOT)
print("FIG_DIR  =", FIG_DIR)
print("step grid", STEP_GRID, "  jets/step", GLOBAL_BATCH)

In [ ]:
HEADER_RE = re.compile(
    r"eta0=(?P<eta0>\S+)\s+arch=L(?P<L>\d+)_D(?P<D>\d+)_h(?P<h>\d+)"
)
VAL_RE = re.compile(r"\[val\] epoch=(?P<epoch>\d+) step=(?P<step>\d+)(?P<rest>.*)")
TRAIN_RE = re.compile(
    r"\[train\] step=(?P<step>\d+) epoch=(?P<epoch>\d+)"
    r" train_loss=(?P<train_loss>[-+eE0-9.]+) train_acc=(?P<train_acc>[-+eE0-9.]+)"
)
KV_RE = re.compile(r"(val_\w+)=([-+eE0-9.]+)")
LOG_NAME_RE = re.compile(r"^(?P<prefix>jc_att(?:_L5)?_s25k)_(?P<job>\d+)_(?P<task>\d+)\.out$")


def _arch_label(depth: int, width: int, heads: int) -> str:
    return rf"$L={int(depth)}$, $D={int(width)}$, $h={int(heads)}$"


def _infer_log_identity(path: Path) -> dict:
    meta = {"eta0": None, "depth": None, "width": None, "heads": None, "source": str(path)}
    m = LOG_NAME_RE.match(path.name)
    if m:
        ldh = LOG_ARCH.get(m.group("prefix"))
        if ldh:
            meta["depth"], meta["width"], meta["heads"] = ldh
        meta["eta0"] = TASK_ETA.get(int(m.group("task")))
        meta["job_id"] = m.group("job")
        meta["task"] = int(m.group("task"))
    with path.open(errors="replace") as fh:
        for line in fh:
            hm = HEADER_RE.search(line)
            if hm:
                meta["eta0"] = float(hm.group("eta0"))
                meta["depth"] = int(hm.group("L"))
                meta["width"] = int(hm.group("D"))
                meta["heads"] = int(hm.group("h"))
                break
    return meta


def _parse_log(path: Path) -> tuple[list[dict], list[dict], dict]:
    ident = _infer_log_identity(path)
    vals, trains = [], []
    with path.open(errors="replace") as fh:
        for line in fh:
            vm = VAL_RE.search(line)
            if vm:
                row = {
                    **{k: ident[k] for k in ("eta0", "depth", "width", "heads")},
                    "epoch": int(vm.group("epoch")),
                    "global_step": int(vm.group("step")),
                    "source": path.name,
                    "origin": "slurm",
                }
                for key, val in KV_RE.findall(vm.group("rest")):
                    row[key] = float(val)
                vals.append(row)
                continue
            tm = TRAIN_RE.search(line)
            if tm:
                trains.append(
                    {
                        **{k: ident[k] for k in ("eta0", "depth", "width", "heads")},
                        "epoch": int(tm.group("epoch")),
                        "global_step": int(tm.group("step")),
                        "train_loss": float(tm.group("train_loss")),
                        "train_acc": float(tm.group("train_acc")),
                        "source": path.name,
                        "origin": "slurm",
                    }
                )
    return vals, trains, ident


def _list_logs() -> list[Path]:
    paths = []
    for glob in LOG_GLOBS:
        paths.extend(SLURM_LOG_ROOT.glob(glob))
    return sorted(paths)


def load_from_logs() -> tuple[pd.DataFrame, pd.DataFrame]:
    vals, trains = [], []
    for path in _list_logs():
        v, t, ident = _parse_log(path)
        if ident["eta0"] is None:
            print("skip unparseable log", path.name)
            continue
        vals.extend(v)
        trains.extend(t)
    val_df = pd.DataFrame(vals)
    train_df = pd.DataFrame(trains)
    keys = ["depth", "width", "heads", "eta0", "global_step"]
    if not val_df.empty:
        val_df = val_df.drop_duplicates(keys, keep="last")
    if not train_df.empty:
        train_df = train_df.drop_duplicates(keys, keep="last")
    return val_df, train_df


def load_from_parquet() -> pd.DataFrame:
    frames = []
    for path in sorted(OUT_ROOT.glob(PARQUET_GLOB)):
        df = pd.read_parquet(path)
        if "val_roc_auc" not in df.columns:
            continue
        sub = df.loc[df["val_roc_auc"].notna()].copy()
        if sub.empty:
            continue
        keep = {
            "eta_0": "eta0",
            "depth": "depth",
            "width": "width",
            "heads": "heads",
            "epoch": "epoch",
            "global_step": "global_step",
            "val_loss": "val_loss",
            "val_acc": "val_acc",
            "val_roc_auc": "val_roc_auc",
            "val_bg_rejection": "val_bg_rejection",
            "train_loss": "train_loss",
            "train_acc": "train_acc",
        }
        sub = sub.rename(columns={c: k for c, k in keep.items() if c in sub.columns})
        sub = sub[[c for c in keep.values() if c in sub.columns]]
        sub["source"] = path.name
        sub["origin"] = "parquet"
        frames.append(sub)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    return out.drop_duplicates(["depth", "width", "heads", "eta0", "global_step"], keep="last")


def merge_val(log_val: pd.DataFrame, pq_val: pd.DataFrame) -> pd.DataFrame:
    frames = [df for df in (log_val, pq_val) if df is not None and not df.empty]
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    # parquet wins on collisions once it starts flushing mid-epoch val
    origin_rank = {"slurm": 0, "parquet": 1}
    out["_rank"] = out["origin"].map(origin_rank).fillna(0)
    out = out.sort_values("_rank").drop_duplicates(
        ["depth", "width", "heads", "eta0", "global_step"], keep="last"
    )
    return out.drop(columns="_rank").sort_values(["depth", "width", "eta0", "global_step"])


def at_step_grid(val: pd.DataFrame, grid: tuple[int, ...] = STEP_GRID) -> pd.DataFrame:
    if val.empty:
        return val
    return val.loc[val["global_step"].isin(grid)].copy()


def status_table(val: pd.DataFrame, train: pd.DataFrame) -> pd.DataFrame:
    rows = []
    keys = ["depth", "width", "heads", "eta0"]
    combos = []
    for df in (val, train):
        if not df.empty:
            combos.append(df[keys].drop_duplicates())
    if not combos:
        return pd.DataFrame()
    runs = pd.concat(combos, ignore_index=True).drop_duplicates()
    for rec in runs.itertuples(index=False):
        mask = lambda d: (
            (d["depth"] == rec.depth)
            & (d["width"] == rec.width)
            & (d["heads"] == rec.heads)
            & (d["eta0"] == rec.eta0)
        )
        vsub = val.loc[mask(val)] if not val.empty else val
        tsub = train.loc[mask(train)] if not train.empty else train
        latest_val_step = int(vsub["global_step"].max()) if len(vsub) else 0
        latest_train_step = int(tsub["global_step"].max()) if len(tsub) else 0
        latest_step = max(latest_val_step, latest_train_step)
        row = {
            "arch": _arch_label(rec.depth, rec.width, rec.heads).replace("$", ""),
            "eta0": rec.eta0,
            "latest_step": latest_step,
            "latest_val_step": latest_val_step,
            "jets_seen_M": round(latest_step * GLOBAL_BATCH / 1e6, 2),
            "frac_of_25k": latest_step / MAX_STEPS,
        }
        if len(vsub):
            last = vsub.loc[vsub["global_step"].idxmax()]
            row["val_roc_latest"] = last.get("val_roc_auc")
            row["val_acc_latest"] = last.get("val_acc")
        for step in STEP_GRID:
            hit = vsub.loc[vsub["global_step"] == step] if len(vsub) else vsub
            row[f"roc@{step // 1000}k"] = (
                float(hit["val_roc_auc"].iloc[0]) if len(hit) else pd.NA
            )
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(["arch", "eta0"])
    return out.reset_index(drop=True)

## Refresh

Re-run this cell to pick up new Slurm logs / parquet after a resubmit.

In [ ]:
log_val, train = load_from_logs()
pq_val = load_from_parquet()
val = merge_val(log_val, pq_val)
grid = at_step_grid(val)

n_logs = len(_list_logs())
n_parquet = len(list(OUT_ROOT.glob(PARQUET_GLOB)))
n_runs = len(list(OUT_ROOT.glob(RUN_DIR_GLOB)))
print(f"logs={n_logs}  parquet={n_parquet}  run dirs={n_runs}")
print(f"val rows={len(val)}  train rows={len(train)}  grid rows={len(grid)}")
if not val.empty:
    print("val steps present:", sorted(val["global_step"].unique().tolist()))
    print("archs:", val[["depth", "width", "heads"]].drop_duplicates().to_dict("records"))
else:
    print("No val rows yet — waiting on the first 2500-step check.")

status = status_table(val, train)
status

## Transfer curves at 5k-step ticks

Color = architecture. Linestyle = optimizer-step horizon. Missing ticks (jobs not there yet) are omitted, not interpolated.

In [ ]:
Y_SPECS = (
    ("val_roc_auc", "val ROC AUC"),
    ("val_acc", "val accuracy"),
    ("val_loss", "val loss"),
)
STEP_LS = {
    5_000: "-",
    10_000: "--",
    15_000: ":",
    20_000: "-.",
    25_000: (0, (3, 1, 1, 1)),
}


def plot_step_transfer(val: pd.DataFrame, *, stem: str = "jetclass_s25k_step_transfer") -> None:
    plot_df = at_step_grid(val)
    if plot_df.empty:
        print("No 5k-grid val points yet. First overlay appears at step 5000.")
        return
    fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.35))
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    archs = (
        plot_df[["depth", "width", "heads"]]
        .drop_duplicates()
        .sort_values(["depth", "width"])
        .itertuples(index=False, name=None)
    )
    arch_handles, step_handles_map = [], {}
    markers = ("o", "s")
    for i, (depth, width, heads) in enumerate(archs):
        color = colors[i % len(colors)]
        marker = markers[i % len(markers)]
        arch_handles.append(
            Line2D(
                [],
                [],
                color=color,
                marker=marker,
                markersize=3,
                linewidth=0.9,
                label=_arch_label(depth, width, heads),
            )
        )
        sub_arch = plot_df[
            (plot_df["depth"] == depth)
            & (plot_df["width"] == width)
            & (plot_df["heads"] == heads)
        ]
        for step, sub in sub_arch.groupby("global_step"):
            step = int(step)
            sub = sub.sort_values("eta0")
            ls = STEP_LS.get(step, "-")
            if step not in step_handles_map:
                step_handles_map[step] = Line2D(
                    [],
                    [],
                    color="0.35",
                    linestyle=ls,
                    linewidth=0.9,
                    label=f"{step // 1000}k steps",
                )
            for ax, (col, _) in zip(axes, Y_SPECS):
                if col not in sub.columns or sub[col].isna().all():
                    continue
                ax.plot(
                    sub["eta0"],
                    sub[col],
                    color=color,
                    marker=marker,
                    linestyle=ls,
                    markersize=3,
                    linewidth=0.9,
                )
    for ax, (_, ylabel) in zip(axes, Y_SPECS):
        ax.set_xscale("log")
        ax.set_xlabel(r"$\eta_0$")
        ax.set_ylabel(ylabel)
        ax.grid(True, which="both", alpha=0.25)
    fig.legend(
        handles=arch_handles + [step_handles_map[s] for s in STEP_GRID if s in step_handles_map],
        loc="upper center",
        ncol=4,
        fontsize=7,
        frameon=False,
        bbox_to_anchor=(0.5, 1.18),
    )
    fig.tight_layout()
    savefig(fig, stem, fig_dir=FIG_DIR)
    plt.show()


plot_step_transfer(val)

## Val ROC vs steps

One panel per architecture, one line per $\eta_0$. Vertical ticks mark the 5k transfer grid. This is the plot that should fill in as gpu-test shots land.

In [ ]:
def plot_val_vs_steps(val: pd.DataFrame, *, stem: str = "jetclass_s25k_val_vs_steps") -> None:
    if val.empty:
        print("No val checkpoints yet.")
        return
    archs = (
        val[["depth", "width", "heads"]]
        .drop_duplicates()
        .sort_values(["depth", "width"])
        .itertuples(index=False, name=None)
    )
    archs = list(archs)
    fig, axes = plt.subplots(1, len(archs), figsize=(3.6 * len(archs), 2.5), squeeze=False)
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    for ax, (depth, width, heads) in zip(axes[0], archs):
        sub_arch = val[
            (val["depth"] == depth) & (val["width"] == width) & (val["heads"] == heads)
        ].sort_values(["eta0", "global_step"])
        etas = sorted(sub_arch["eta0"].unique())
        for j, eta0 in enumerate(etas):
            sub = sub_arch.loc[sub_arch["eta0"] == eta0]
            ax.plot(
                sub["global_step"],
                sub["val_roc_auc"],
                marker="o",
                markersize=3,
                linewidth=0.9,
                color=colors[j % len(colors)],
                label=rf"$\eta_0={eta0:g}$",
            )
        for step in STEP_GRID:
            ax.axvline(step, color="0.85", linewidth=0.6, zorder=0)
        ax.set_xlim(0, MAX_STEPS * 1.02)
        ax.set_xlabel("optimizer step")
        ax.set_ylabel("val ROC AUC")
        ax.set_title(_arch_label(depth, width, heads), fontsize=9)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=6.5, frameon=False, loc="lower right")
    fig.tight_layout()
    savefig(fig, stem, fig_dir=FIG_DIR)
    plt.show()


plot_val_vs_steps(val)

## Leading $\eta_0$ at each horizon

Highest val ROC AUC among completed 5k ticks. Use this as the go / no-go for a long (expensive) run once 15k–25k is in.

In [ ]:
def leading_eta(val: pd.DataFrame) -> pd.DataFrame:
    plot_df = at_step_grid(val)
    if plot_df.empty:
        return plot_df
    rows = []
    for (depth, width, heads, step), sub in plot_df.groupby(
        ["depth", "width", "heads", "global_step"]
    ):
        sub = sub.dropna(subset=["val_roc_auc"]).sort_values("val_roc_auc", ascending=False)
        if sub.empty:
            continue
        best = sub.iloc[0]
        rows.append(
            {
                "arch": _arch_label(depth, width, heads).replace("$", ""),
                "step": int(step),
                "jets_M": round(int(step) * GLOBAL_BATCH / 1e6, 2),
                "eta0_star": best["eta0"],
                "val_roc_auc": best["val_roc_auc"],
                "val_acc": best.get("val_acc"),
                "n_eta": len(sub),
            }
        )
    return pd.DataFrame(rows).sort_values(["arch", "step"]).reset_index(drop=True)


stars = leading_eta(val)
stars